# tSNR-stratified robustness figure

Robustness-tier check (CLAUDE.md, "tSNR stratification") on claim 2: does
connectome similarity depend on acquisition signal quality, and does the
within-task > between-task ordering survive when tSNR is held high? Reads
`output_data/tsnr_strata/tsnr_strata.tsv` (written by `run-tsnr-strata`) and
plots only — no similarity computation here, which lives in
`analysis/tsnr_strata.py`.

**Standalone figure, deliberately not placed in `connectome_figure.svg`** —
the domain panels' placement there was an explicit, user-requested exception
and does not generalize (CLAUDE.md).

This notebook renders two panels, one per `stratum_def` — the two ways of
defining the tSNR stratum:

- **`raw`** (`tsnr_bins_raw.png`) — median split on `tsnr` itself. Directly
  interpretable, and the reported/primary result.
- **`fd_residual`** (`tsnr_bins_fd_residual.png`) — median split on `tsnr`
  residualized on `fd_mean` within the same (subject, dataset) cell. tSNR and
  `fd_mean` correlate at r=-0.68 — related, but not equivalent — so this is
  kept as a separate **sensitivity-analysis** figure, checking whether the
  `raw` result holds up once head motion is regressed out, rather than the
  headline. Not used for reporting on its own.

Both panels are drawn with within-task bars grouped together and between-task
bars grouped together (task outer, tSNR-pairing inner), so the large task
effect reads as two clean blocks rather than interleaving with the much
smaller tSNR effect.

Whole-brain tSNR only: `tables/atlas_tsnr/` is populated upstream for `floc`,
`retinotopy` and `things` alone — exactly the three datasets the 1800 s gate
removes — so the per-network `tsnr_{network}` columns are non-NaN for none of
the covered sessions.

The duration/tSNR/motion balance audit and the permutation-test effect sizes
are still computed by `run-tsnr-strata` into `tsnr_balance.tsv` and
`tsnr_permutation.tsv`; they are reported as text (key stats and ranges), not
as figure panels here.

Panels:
1. `tsnr_bins_raw.png` — the six high/high, low/high, low/low x within/between-task
   bins, one group of bars per network, "cell" split, `raw` definition.
2. `tsnr_bins_fd_residual.png` — same layout, `fd_residual` definition.
3. `tsnr_bins_legend.png` — shared legend for both panels.


In [1]:
import os
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import yaml
from airoh.figures import panel_size

FIGURE_DPI = int(os.environ.get("FIGURE_MONTAGE_DPI", 300))

output_dir = Path(os.environ.get("OUTPUT_DATA_DIR", "../output_data")).resolve()
figures_base = Path(os.environ.get("FIGURES_DIR", output_dir / "figures")).resolve()

project_root = output_dir.parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

figure_dir = figures_base / "figure_tsnr"
figure_dir.mkdir(parents=True, exist_ok=True)

with open(project_root / "invoke.yaml") as handle:
    invoke_config = yaml.safe_load(handle)

PARCELLATION = invoke_config["parcellation"]
NETWORK_ORDER = invoke_config["parcellations"][PARCELLATION]["network_order"]
MEASURE = invoke_config.get("analysis_measure", "pearson")

tsnr_dir = output_dir / "tsnr_strata"
tsnr_bins = pd.read_csv(tsnr_dir / "tsnr_strata.tsv", sep="\t")

print(f"\U0001f4c2 {PARCELLATION}, measure={MEASURE}: {len(tsnr_bins)} tSNR-bin rows")

📂 cneuromod2026, measure=pearson: 216 tSNR-bin rows


In [2]:
def save_legend(handles, labels, name, default_size, ncol=None, fontsize=7):
    """Render `handles`/`labels` alone into `{name}` as a horizontal strip."""
    if not handles:
        return
    figsize = panel_size(f"figure_tsnr/{name}", default_size)
    fig = plt.figure(figsize=figsize, layout="constrained")
    fig.legend(
        handles, labels, loc="center",
        ncol=ncol or min(len(handles), 5), fontsize=fontsize, frameon=False,
    )
    fig.savefig(figure_dir / name, dpi=FIGURE_DPI)
    plt.close(fig)
    print(f"\u2705 wrote {figure_dir / name} at {figsize} in")

In [3]:
# Shared color scheme, deliberately the same system as figure_motion.ipynb so
# the two QC-axis figures read as one pair: hue = task (categorical, 2 series,
# validated blue/orange), alpha = stratum pairing as an ordinal *quality*
# ramp — more opaque means worse acquisition quality. On the motion axis that
# was low-low -> high-high; here it is high-high -> low-low, because high tSNR
# is the good end. So opacity carries the same meaning across both figures
# even though the labels invert.
TASK_HUE = {"within-task": "#2a78d6", "between-task": "#eb6834"}
TSNR_ALPHA = {"high-high": 0.40, "low-high": 0.70, "low-low": 1.0}
TSNR_ORDER = ["high-high", "low-high", "low-low"]
TASK_ORDER = ["within-task", "between-task"]
# Task outer, tSNR-pairing inner: groups all within-task bars together and all
# between-task bars together, so the (much larger) task effect reads as two
# clean blocks instead of interleaving with the (tiny) tSNR effect.
BIN_ORDER = [f"{s}/{t}" for t in TASK_ORDER for s in TSNR_ORDER]

# `raw` is the primary, reported result — directly interpretable as a median
# split on tSNR itself. `fd_residual` (median split on tSNR residualized on
# fd_mean within cell) is kept as a separate sensitivity-analysis figure: tSNR
# and fd_mean correlate at r=-0.68, so tSNR and motion are related but not
# equivalent, and the residualized definition checks whether the raw-tSNR
# result is really doing something beyond what the motion figure already
# shows — it is not the headline, so it gets its own panel rather than
# crowding the primary one.
DEFINITIONS = ["raw", "fd_residual"]
DEFINITION_TITLES = {
    "raw": "raw tSNR",
    "fd_residual": "tSNR residualized on fd_mean (within cell) — sensitivity analysis",
}


def bin_color(bin_label):
    stratum, task = bin_label.split("/")
    return TASK_HUE[task], TSNR_ALPHA[stratum]

In [4]:
# Two separate panels — tsnr_bins_raw.png (primary) and
# tsnr_bins_fd_residual.png (sensitivity analysis) — rather than one figure
# with two rows: `raw` is the reported result, `fd_residual` is a check on it,
# and keeping them as separate files makes that asymmetry explicit instead of
# implying two equally-weighted results. Both share one legend
# (tsnr_bins_legend.png), since the six bins are identical across panels.
cell_bins = tsnr_bins[tsnr_bins["split"] == "cell"] if len(tsnr_bins) else tsnr_bins

x = np.arange(len(NETWORK_ORDER))
width = 0.13
legend_handles, legend_labels = [], []

for definition, filename in [("raw", "tsnr_bins_raw.png"),
                              ("fd_residual", "tsnr_bins_fd_residual.png")]:
    figsize = panel_size(f"figure_tsnr/{filename}", (6.0, 4.5))
    fig, ax = plt.subplots(figsize=figsize, layout="constrained")

    subset = cell_bins[cell_bins["stratum_def"] == definition] if len(cell_bins) else cell_bins
    if len(subset):
        for i, bin_label in enumerate(BIN_ORDER):
            color, alpha = bin_color(bin_label)
            values = []
            for network in NETWORK_ORDER:
                row = subset[(subset["network"] == network) & (subset["bin"] == bin_label)]
                values.append(row["median"].iloc[0] if len(row) else np.nan)
            bars = ax.bar(x + (i - 2.5) * width, values, width, color=color, alpha=alpha,
                          edgecolor="white", linewidth=0.4)
            if definition == "raw":
                legend_handles.append(bars[0])
                legend_labels.append(bin_label)
        ax.set_xticks(x)
        ax.set_xticklabels(NETWORK_ORDER, rotation=45, ha="right", fontsize=7)
        ax.set_ylabel("median similarity (Fisher-z)")
        ax.set_title(DEFINITION_TITLES[definition], fontsize=8, loc="left")
        ax.spines[["top", "right"]].set_visible(False)
    else:
        ax.text(0.5, 0.5, "no QC-covered sessions\n(smoke run, or too few cells)",
                ha="center", va="center", transform=ax.transAxes, color="0.5", fontsize=8)
        ax.set_xticks([])
        ax.set_yticks([])

    fig.savefig(figure_dir / filename, dpi=FIGURE_DPI)
    plt.close(fig)
    print(f"✅ wrote {figure_dir / filename} at {figsize} in")

save_legend(legend_handles, legend_labels, "tsnr_bins_legend.png", (6.0, 0.9), ncol=3)

✅ wrote /home/pbellec/git/cneuromod.all.connectome_stats/output_data/figures/figure_tsnr/tsnr_bins_raw.png at (6.0, 4.5) in
✅ wrote /home/pbellec/git/cneuromod.all.connectome_stats/output_data/figures/figure_tsnr/tsnr_bins_fd_residual.png at (6.0, 4.5) in
✅ wrote /home/pbellec/git/cneuromod.all.connectome_stats/output_data/figures/figure_tsnr/tsnr_bins_legend.png at (6.0, 0.9) in
